### Comparative Evaluation of Deep Learning Models for Brain Stroke Detection using Neuroimaging Data

#### Imports and Library Configs
##### Core Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
import warnings

warnings.filterwarnings("ignore")

##### Deep Learning Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # type: ignore
from tensorflow.keras.applications import VGG16, DenseNet201, ResNet50, EfficientNetB0  # type: ignore
from tensorflow.keras.models import Model  # type: ignore
from tensorflow.keras.layers import (  # type: ignore
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization,
)
from tensorflow.keras.optimizers import Adam  # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint  # type: ignore
from tensorflow.keras.regularizers import l2  # type: ignore  # type: ignore

##### Metrics and Evaluation Libraries

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

##### Random Seed and Plot Config

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

#### Data Directory Setup

In [ ]:
def setup_data_directories():
    base_dir = "dataset"
    train_dir = os.path.join(base_dir, "Training")
    test_dir = os.path.join(base_dir, "Testing")

    print("=" * 60)
    print("DATA DIRECTORY STRUCTURE")
    print("=" * 60)

    # Check training directory
    if not os.path.exists(train_dir):
        print(f"⚠️ Warning: Training directory not found: {train_dir}")
    else:
        train_classes = os.listdir(train_dir)
        print(f"✅ Training directory found. Classes: {train_classes}")

    # Check testing directory
    if not os.path.exists(test_dir):
        print(f"⚠️ Warning: Testing directory not found: {test_dir}")
    else:
        test_classes = os.listdir(test_dir)
        print(f"✅ Testing directory found. Classes: {test_classes}")

    return train_dir, test_dir


# Execute directory check
train_dir, test_dir = setup_data_directories()

#### Data Augmentation and Generators

In [ ]:
def create_data_generators(train_dir, test_dir, img_size=(224, 224), batch_size=32):

    # Training data augmentation (helps prevent overfitting)
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,  # Normalize pixel values
        rotation_range=20,  # Random rotation
        width_shift_range=0.2,  # Horizontal shift
        height_shift_range=0.2,  # Vertical shift
        shear_range=0.2,  # Shear transformation
        zoom_range=0.2,  # Random zoom
        horizontal_flip=True,  # Horizontal flip
        vertical_flip=False,  # No vertical flip (medical images)
        brightness_range=[0.8, 1.2],  # Brightness adjustment
        fill_mode="nearest",  # Fill missing pixels
        validation_split=0.2,  # 20% for validation
    )

    # Test/Validation data (no augmentation, only rescaling)
    test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

    # Training data (80% of training directory)
    train_data = train_datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="binary",
        subset="training",
        shuffle=True,
        seed=42,
    )

    # Validation data (20% of training directory)
    val_data = train_datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="binary",
        subset="validation",
        shuffle=True,
        seed=42,
    )

    # Test data (no shuffling for evaluation)
    test_data = test_datagen.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="binary",
        shuffle=False,
    )

    # Print dataset statistics
    print("\n" + "=" * 60)
    print("DATASET STATISTICS")
    print("=" * 60)
    print(f"Training samples: {train_data.samples}")
    print(f"Validation samples: {val_data.samples}")
    print(f"Testing samples: {test_data.samples}")
    print(f"Number of classes: {len(train_data.class_indices)}")
    print(f"Class labels: {train_data.class_indices}")

    return train_data, val_data, test_data


# Create the data generators
train_data, val_data, test_data = create_data_generators(train_dir, test_dir)

#### Sample Image

In [ ]:
def visualize_data_samples(data_generator, title="Sample Images", num_samples=8):
    images, labels = next(data_generator)

    # Calculate grid dimensions
    cols = 4
    rows = (num_samples + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
    axes = axes.ravel() if rows > 1 else [axes] if cols == 1 else axes.ravel()

    class_names = list(data_generator.class_indices.keys())

    for i in range(min(num_samples, len(images))):
        axes[i].imshow(images[i])

        # Get class label
        label_val = labels[i] if np.isscalar(labels[i]) else labels[i][0]
        class_label = class_names[int(label_val)]

        axes[i].set_title(f"Class: {class_label}", fontsize=10, fontweight="bold")
        axes[i].axis("off")

    # Hide unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis("off")

    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Display sample images
visualize_data_samples(train_data, "Training Sample Images")
visualize_data_samples(test_data, "Testing Sample Images")

#### Class Distribution

In [ ]:
def plot_data_distribution(train_data, val_data, test_data):
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    datasets = [
        ("Training", train_data.classes),
        ("Validation", val_data.classes),
        ("Testing", test_data.classes),
        (
            "Total",
            np.concatenate([train_data.classes, val_data.classes, test_data.classes]),
        ),
    ]

    colors = ["skyblue", "lightcoral"]
    class_names = list(train_data.class_indices.keys())

    for idx, (name, classes) in enumerate(datasets):
        unique, counts = np.unique(classes, return_counts=True)

        # Bar plot
        axes[idx].bar(unique, counts, color=colors, alpha=0.8, edgecolor="black")
        axes[idx].set_title(f"{name} Set", fontweight="bold")
        axes[idx].set_xlabel("Class")
        axes[idx].set_ylabel("Count")
        axes[idx].set_xticks(unique)
        axes[idx].set_xticklabels(class_names)
        axes[idx].grid(True, alpha=0.3, axis="y")

        # Add percentage labels
        total = sum(counts)
        for i, (cls, count) in enumerate(zip(unique, counts)):
            percentage = (count / total) * 100
            axes[idx].text(
                cls,
                count + max(counts) * 0.02,
                f"{percentage:.1f}%",
                ha="center",
                fontsize=9,
                fontweight="bold",
            )

    plt.suptitle("Class Distribution Analysis", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Display class distribution
plot_data_distribution(train_data, val_data, test_data)

#### Model Building Function

In [ ]:
def build_transfer_learning_model(
    base_model_name="DenseNet201",
    input_shape=(224, 224, 3),
    num_classes=1,
    dropout_rate=0.5,
    l2_reg=0.001,
):
    # Available base models
    base_models = {
        "VGG16": VGG16,
        "DenseNet201": DenseNet201,
        "ResNet50": ResNet50,
        "EfficientNetB0": EfficientNetB0,
    }

    if base_model_name not in base_models:
        raise ValueError(
            f"Model {base_model_name} not supported. Choose from {list(base_models.keys())}"
        )

    print(f"\n🏗️ Building model with {base_model_name} backbone...")

    # Load pre-trained model (without top classification layer)
    base_model = base_models[base_model_name](
        weights="imagenet", include_top=False, input_shape=input_shape
    )

    # Freeze base model layers (initially)
    for layer in base_model.layers:
        layer.trainable = False

    # Build custom classification head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)  # Reduce spatial dimensions
    x = BatchNormalization()(x)  # Normalize activations
    x = Dense(256, activation="relu", kernel_regularizer=l2(l2_reg))(x)
    x = Dropout(dropout_rate)(x)  # Prevent overfitting
    x = BatchNormalization()(x)
    x = Dense(128, activation="relu", kernel_regularizer=l2(l2_reg))(x)
    x = Dropout(dropout_rate)(x)

    # Output layer (binary or multi-class)
    if num_classes == 1:
        output = Dense(1, activation="sigmoid")(x)  # Binary classification
        loss = "binary_crossentropy"
        metrics = [
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ]
    else:
        output = Dense(num_classes, activation="softmax")(x)  # Multi-class
        loss = "categorical_crossentropy"
        metrics = ["accuracy"]

    # Create and compile model
    model = Model(inputs=base_model.input, outputs=output)
    optimizer = Adam(learning_rate=0.0001)
    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

    return model, base_model_name

#### Training Callbacks

In [ ]:
def create_callbacks(model_name):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Create directories if they don't exist
    os.makedirs("models", exist_ok=True)
    os.makedirs("logs", exist_ok=True)

    callbacks = [
        # Stop training when validation loss stops improving
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1,
            mode="min",
        ),
        # Reduce learning rate when plateau is detected
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1,
            mode="min",
        ),
        # Save the best model checkpoint
        ModelCheckpoint(
            filepath=f"models/{model_name}_{timestamp}_best.h5",
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1,
        ),
    ]

    return callbacks

#### Model Evaluation Function

In [ ]:
def evaluate_model(model, test_data, model_name, return_predictions=True):

    print("\n" + "=" * 60)
    print(f"📊 EVALUATING: {model_name}")
    print("=" * 60)

    # Get predictions
    y_pred_proba = model.predict(test_data, verbose=0)
    y_pred = (y_pred_proba > 0.5).astype(int)
    y_true = test_data.classes

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")
    roc_auc = roc_auc_score(y_true, y_pred_proba)

    # Calculate average precision
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_proba)
    avg_precision = np.mean(precision_vals)

    # Print results
    print(f"✅ Accuracy:  {accuracy:.4f}")
    print(f"✅ Precision: {precision:.4f}")
    print(f"✅ Recall:    {recall:.4f}")
    print(f"✅ F1-Score:  {f1:.4f}")
    print(f"✅ ROC-AUC:   {roc_auc:.4f}")
    print(f"✅ Avg Prec:  {avg_precision:.4f}")

    # Create metrics dictionary
    metrics_dict = {
        "model_name": model_name,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "average_precision": float(avg_precision),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    if return_predictions:
        return metrics_dict, y_true, y_pred, y_pred_proba
    else:
        return metrics_dict

#### Training and Evaluation Loop

In [ ]:
# Define model architectures to test
architectures = ["DenseNet201", "VGG16", "ResNet50", "EfficientNetB0"]

# Storage for results
models_metrics = []
all_histories = []
model_names = []
y_true_list = []
y_pred_list = []
y_pred_proba_list = []

print("\n" + "=" * 60)
print("🤖 MODEL TRAINING & EVALUATION")
print("=" * 60)

for arch in architectures:
    print(f"\n▶️ Training {arch}...")

    try:
        # Build model
        model, model_name = build_transfer_learning_model(
            base_model_name=arch,
            input_shape=(224, 224, 3),
            num_classes=1,
            dropout_rate=0.5,
            l2_reg=0.001,
        )

        # Create callbacks
        callbacks = create_callbacks(model_name)

        # Train model (silent mode for cleaner output)
        history = model.fit(
            train_data,
            validation_data=val_data,
            epochs=20,
            callbacks=callbacks,
            verbose=0,
        )

        all_histories.append(history)
        model_names.append(model_name)

        # Evaluate model
        metrics, y_true, y_pred, y_pred_proba = evaluate_model(
            model, test_data, model_name, return_predictions=True
        )

        models_metrics.append(metrics)
        y_true_list.append(y_true)
        y_pred_list.append(y_pred)
        y_pred_proba_list.append(y_pred_proba)

        # Save training history
        history_df = pd.DataFrame(history.history)
        history_df.to_csv(f"models/{model_name}_history.csv", index=False)

        print(f"✅ {model_name} completed successfully")

    except Exception as e:
        print(f"❌ Error training {arch}: {str(e)}")
        continue

#### Training History

In [ ]:
def plot_training_comparison(all_histories, model_names):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    metrics = ["loss", "accuracy", "auc", "precision", "recall", "lr"]

    # Color palette for different models
    colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

    for idx, metric in enumerate(metrics):
        ax = axes[idx // 3, idx % 3]

        for i, history in enumerate(all_histories):
            if metric == "lr":
                if "lr" in history.history:
                    ax.plot(
                        history.history["lr"],
                        label=model_names[i],
                        linewidth=2,
                        color=colors[i],
                    )
                ax.set_yscale("log")
            elif metric in history.history:
                # Training metric
                ax.plot(
                    history.history[metric],
                    label=f"{model_names[i]} (Train)",
                    linewidth=2,
                    alpha=0.7,
                    color=colors[i],
                )
                # Validation metric
                val_metric = f"val_{metric}"
                if val_metric in history.history:
                    ax.plot(
                        history.history[val_metric],
                        label=f"{model_names[i]} (Val)",
                        linewidth=2,
                        linestyle="--",
                        alpha=0.7,
                        color=colors[i],
                    )

        ax.set_title(f"{metric.upper()}", fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric.capitalize())
        ax.legend(fontsize=8, loc="best")
        ax.grid(True, alpha=0.3)

    plt.suptitle("Model Training Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Plot training comparison if we have results
if all_histories:
    plot_training_comparison(all_histories, model_names)

#### Confusion Matrix

In [ ]:
def plot_confusion_matrices(y_true_list, y_pred_list, model_names, normalize=True):
    n_models = len(model_names)
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))

    if n_models == 1:
        axes = [axes]

    for idx, (y_true, y_pred, name) in enumerate(
        zip(y_true_list, y_pred_list, model_names)
    ):
        cm = confusion_matrix(y_true, y_pred)

        if normalize:
            cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
            fmt = ".2f"
        else:
            fmt = "d"

        # Create heatmap
        # im = axes[idx].imshow(cm, interpolation='nearest', cmap='Blues')
        axes[idx].set_title(f"{name}", fontweight="bold")

        # Add text annotations
        thresh = cm.max() / 2.0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                axes[idx].text(
                    j,
                    i,
                    format(cm[i, j], fmt),
                    ha="center",
                    va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontweight="bold",
                )

        axes[idx].set_xlabel("Predicted")
        axes[idx].set_ylabel("True")
        axes[idx].set_xticks([0, 1])
        axes[idx].set_yticks([0, 1])
        axes[idx].set_xticklabels(["Normal", "Stroke"])
        axes[idx].set_yticklabels(["Normal", "Stroke"])

    plt.suptitle("Confusion Matrix Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Plot confusion matrices
if y_true_list:
    plot_confusion_matrices(y_true_list, y_pred_list, model_names)

#### ROC Curve

In [ ]:
def plot_roc_comparison(y_true_list, y_pred_proba_list, model_names):
    plt.figure(figsize=(10, 8))

    colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

    for i, (y_true, y_pred_proba, name) in enumerate(
        zip(y_true_list, y_pred_proba_list, model_names)
    ):
        fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr, tpr, color=colors[i], lw=2.5, label=f"{name} (AUC = {roc_auc:.3f})"
        )

    # Diagonal line (random classifier)
    plt.plot(
        [0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random Classifier"
    )

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate", fontweight="bold")
    plt.ylabel("True Positive Rate", fontweight="bold")
    plt.title("ROC Curve Comparison", fontsize=14, fontweight="bold")
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Plot ROC curves
if y_true_list:
    plot_roc_comparison(y_true_list, y_pred_proba_list, model_names)

#### Precision-Recall Curve

In [ ]:
def plot_precision_recall_comparison(y_true_list, y_pred_proba_list, model_names):
    plt.figure(figsize=(10, 8))

    colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

    for i, (y_true, y_pred_proba, name) in enumerate(
        zip(y_true_list, y_pred_proba_list, model_names)
    ):
        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
        avg_precision = np.mean(precision)

        plt.plot(
            recall,
            precision,
            color=colors[i],
            lw=2.5,
            label=f"{name} (AP = {avg_precision:.3f})",
        )

    plt.xlabel("Recall", fontweight="bold")
    plt.ylabel("Precision", fontweight="bold")
    plt.title("Precision-Recall Curve Comparison", fontsize=14, fontweight="bold")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Plot Precision-Recall curves
if y_true_list:
    plot_precision_recall_comparison(y_true_list, y_pred_proba_list, model_names)

#### Prediction Distribution

In [ ]:
def plot_prediction_distribution_comparison(
    y_true_list, y_pred_proba_list, model_names
):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

    # Plot 1: Distribution for class 0 (Normal)
    for i, (y_true, y_pred_proba, name) in enumerate(
        zip(y_true_list, y_pred_proba_list, model_names)
    ):
        y_pred_proba = y_pred_proba.flatten()
        mask = y_true == 0
        if np.any(mask):
            axes[0, 0].hist(
                y_pred_proba[mask], bins=30, alpha=0.5, label=name, color=colors[i]
            )
    axes[0, 0].set_title("Predictions for Class 0 (Normal)", fontweight="bold")
    axes[0, 0].set_xlabel("Predicted Probability")
    axes[0, 0].set_ylabel("Frequency")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Distribution for class 1 (Stroke)
    for i, (y_true, y_pred_proba, name) in enumerate(
        zip(y_true_list, y_pred_proba_list, model_names)
    ):
        y_pred_proba = y_pred_proba.flatten()
        mask = y_true == 1
        if np.any(mask):
            axes[0, 1].hist(
                y_pred_proba[mask], bins=30, alpha=0.5, label=name, color=colors[i]
            )
    axes[0, 1].set_title("Predictions for Class 1 (Stroke)", fontweight="bold")
    axes[0, 1].set_xlabel("Predicted Probability")
    axes[0, 1].set_ylabel("Frequency")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Box plot comparison
    data = [y_pred_proba.flatten() for y_pred_proba in y_pred_proba_list]
    box = axes[1, 0].boxplot(data, labels=model_names, patch_artist=True)
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
    axes[1, 0].set_title("Prediction Distribution - Box Plot", fontweight="bold")
    axes[1, 0].set_ylabel("Predicted Probability")
    axes[1, 0].grid(True, alpha=0.3, axis="y")

    # Plot 4: Violin plot
    parts = axes[1, 1].violinplot(data, showmeans=True, showmedians=True)
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
    axes[1, 1].set_xticks(range(1, len(model_names) + 1))
    axes[1, 1].set_xticklabels(model_names, rotation=45)
    axes[1, 1].set_title("Prediction Distribution - Violin Plot", fontweight="bold")
    axes[1, 1].set_ylabel("Predicted Probability")
    axes[1, 1].grid(True, alpha=0.3, axis="y")

    plt.suptitle("Prediction Distribution Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Plot prediction distributions
if y_true_list:
    plot_prediction_distribution_comparison(y_true_list, y_pred_proba_list, model_names)

#### Best Model Selection

In [ ]:
def select_best_model(models_metrics):

    # Create comparison DataFrame
    comparison_data = []
    for metrics in models_metrics:
        comparison_data.append(
            {
                "Model": metrics["model_name"],
                "Accuracy": metrics["accuracy"],
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "F1-Score": metrics["f1_score"],
                "ROC-AUC": metrics["roc_auc"],
                "Avg_Precision": metrics["average_precision"],
            }
        )

    df_comparison = pd.DataFrame(comparison_data)

    # Find best model by Accuracy
    best_model_idx = df_comparison["Accuracy"].idxmax()
    best_model = df_comparison.loc[best_model_idx]

    # Display results
    print("\n" + "=" * 60)
    print("📊 ALL MODELS PERFORMANCE SUMMARY")
    print("=" * 60)
    print(df_comparison.to_string(index=False))

    print("\n" + "=" * 60)
    print("🏆 BEST PERFORMING MODEL")
    print("=" * 60)
    print(f"Model:       {best_model['Model']}")
    print(f"Accuracy:    {best_model['Accuracy']:.4f}")
    print(f"Precision:   {best_model['Precision']:.4f}")
    print(f"Recall:      {best_model['Recall']:.4f}")
    print(f"F1-Score:    {best_model['F1-Score']:.4f}")
    print(f"ROC-AUC:     {best_model['ROC-AUC']:.4f}")

    # Save comparison to CSV
    df_comparison.to_csv("models/model_comparison.csv", index=False)
    print("\n✅ Comparison saved to 'models/model_comparison.csv'")

    return best_model, df_comparison


# Select and display best model
if models_metrics:
    best_model, comparison_df = select_best_model(models_metrics)

#### Summary

In [ ]:
print("\n" + "=" * 70)
print(" COMPARATIVE EVALUATION OF DEEP LEARNING MODELS")
print(" FOR BRAIN STROKE DETECTION")
print("=" * 70)

print("\n📁 DATASET SUMMARY:")
print(f"   • Training images:   {train_data.samples}")
print(f"   • Validation images: {val_data.samples}")
print(f"   • Testing images:    {test_data.samples}")
print(f"   • Classes:           {list(train_data.class_indices.keys())}")

print("\n🤖 MODELS EVALUATED:")
for name in model_names:
    print(f"   • {name}")

print("\n📊 BEST MODEL RESULTS:")
if models_metrics:
    print(f"   • Model:       {best_model['Model']}")
    print(
        f"   • Accuracy:    {best_model['Accuracy']:.4f} ({best_model['Accuracy'] * 100:.2f}%)"
    )
    print(f"   • Precision:   {best_model['Precision']:.4f}")
    print(f"   • Recall:      {best_model['Recall']:.4f}")
    print(f"   • F1-Score:    {best_model['F1-Score']:.4f}")
    print(f"   • ROC-AUC:     {best_model['ROC-AUC']:.4f}")

print("\n📁 OUTPUT FILES GENERATED:")
print("   • models/*.h5 - Trained model files")
print("   • models/*_history.csv - Training history logs")
print("   • models/model_comparison.csv - Performance comparison table")
print("   • models/best_model_info.txt - Best model details")

print("\n" + "=" * 70)
print("✅ ANALYSIS COMPLETE!")
print("=" * 70)